# Notebook 01: Enterprise Document Exploration & Ingestion
### Project: Enterprise Document Intelligence & RAG Assistant
**Objective:**
Inspect, load, clean, and extract metadata from enterprise PDF and DOCX documents (Leave Policy, Employee Handbook, WFH Policy, Attendance, Benefits, and Code of Conduct).

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.document_loader import DocumentLoader

# Initialize Document Loader
loader = DocumentLoader()
docs_path = os.path.abspath('../data/documents')
print(f"Scanning directory: {docs_path}")
print("Available files:", os.listdir(docs_path) if os.path.exists(docs_path) else "Directory not created yet")

### 1. Ingest All Supported Enterprise Documents
We load both PDF and DOCX files, preserving page numbers, section headers, and document hashes.

In [ ]:
documents = loader.load_directory(docs_path)
print(f"Successfully loaded {len(documents)} document units / pages.")

# Inspect first 3 extracted document objects
for idx, doc in enumerate(documents[:3], 1):
    print(f"\n--- Document Unit #{idx} ---")
    print(f"Filename: {doc.metadata['filename']}")
    print(f"Page: {doc.metadata.get('page_number', 'N/A')}")
    print(f"Section: {doc.metadata.get('section', 'General')}")
    print(f"Character Length: {len(doc.content)}")
    print("Preview:", doc.content[:200].replace('\n', ' '), "...")

### 2. Document Character & Word Distribution
Analyze lengths across different policy files to determine optimal chunk sizes.

In [ ]:
import pandas as pd

records = [
    {
        "filename": d.metadata["filename"],
        "page": d.metadata.get("page_number", 1),
        "section": d.metadata.get("section", "General"),
        "char_count": len(d.content),
        "word_count": len(d.content.split()),
    }
    for d in documents
]

df_stats = pd.DataFrame(records)
display(df_stats.groupby("filename").agg({"char_count": ["count", "mean", "sum"], "word_count": "sum"}))